# Lenormand B4 — Final Three-Fold Test Inference & Submission

这是最终冲榜 notebook，**不重训 14B/27B**。它复用已完成的 adapters：

```text
Factor: 3-fold Qwen3-14B + 3-fold Qwen3.8-27B
        └─ 25% / 75% frozen probability blend + all-OOF thresholds

Task1: 3-fold Qwen3.8-27B Document → Evidence → Conditioned Risk
       └─ three-fold margin ensemble + all-OOF B4-E2 calibrator

Output: row_id, risk_level, semicolon-separated verbatim evidence, factors
```

所有大模型评分按 fold 保存 chunk，Colab 断线后重跑对应格即可续跑。建议 A100 80GB；全部 test 推理可能需要多个 session。


In [ ]:
#@title 0A. 安装冻结环境
%%capture
!pip install -q -U \
  "numpy==2.3.4" "scipy==1.16.3" "pandas==2.3.3" \
  "scikit-learn==1.7.2" "joblib==1.5.2" \
  "transformers>=5.8.0" "accelerate>=1.6.0" "peft>=0.17.0" \
  "bitsandbytes>=0.46.0" "sentence-transformers>=3.4.0" \
  "sentencepiece>=0.2.0" "openpyxl>=3.1.0" "kernels"
!pip install -q -U "flash-linear-attention[cuda]"
!pip install -q -U causal-conv1d --no-build-isolation
print('安装完成：Runtime → Restart session，然后从第 1 格继续。')


> 新 runtime 必须重启一次，让 Qwen3.8 重新探测 FLA / causal-conv1d。重启后不要重跑 0A。


In [ ]:
#@title 1. Drive、路径和续跑开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, os, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
TEST_PATH = ROOT / 'leaderboard.xlsx'
FOLD_REFERENCE = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'

Q14_ROOT = ROOT / 'results' / 'B4P_AVC_FAST3'
Q14_OOF = Q14_ROOT / 'B4P_CORE_OOF.npz'
Q14_DECISION = Q14_ROOT / 'OOF_EVALUATION' / 'B4P_DECISION.json'
Q38_FACTOR_ROOT = (ROOT / 'results' / 'B4_Q38F_FULL64_KERNEL_FOLD0' /
                   'FULL64_FACTOR_FOLD0')
Q38_FACTOR_REPORT = ROOT / 'results' / 'B4_Q38F_FULL64_THREE_FOLD_OOF'
Q38_FACTOR_OOF = Q38_FACTOR_REPORT / 'Q38_FULL64_OOF.npz'
Q38_FACTOR_CONFIG = Q38_FACTOR_REPORT / 'FULL64_CONFIG.json'

TASK1_FOLD0_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_FOLD0'
TASK1_CONFIRM_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_OUTER_CONFIRM'
TASK1_SELECTED_CONFIG = TASK1_FOLD0_ROOT / 'TASK1_CONFIG_SELECTED.json'
FINAL_ROOT = ROOT / 'results' / 'B4_FINAL_SUBMISSION'
META_FINAL_ROOT = FINAL_ROOT / 'META_ALL_OOF_REFIT'
FACTOR_TEST_ROOT = FINAL_ROOT / 'FACTOR_TEST'
TASK1_TEST_ROOT = FINAL_ROOT / 'TASK1_TEST'
SUBMISSION_ROOT = FINAL_ROOT / 'SUBMISSION'

# 全部开启也可续跑。若想一个 session 一折，改成 (0,) / (1,) / (2,)。
RUN_META_REFIT = True
RUN_FACTOR_Q38_FOLDS = (0, 1, 2)
RUN_FACTOR_Q14_FOLDS = (0, 1, 2)
RUN_TASK1_FOLDS = (0, 1, 2)
RUN_FINAL_ASSEMBLY = True

for values in (RUN_FACTOR_Q38_FOLDS, RUN_FACTOR_Q14_FOLDS, RUN_TASK1_FOLDS):
    assert set(values).issubset({0, 1, 2})
FINAL_ROOT.mkdir(parents=True, exist_ok=True)

MODULES = {
    'b1_experiments.py': None,
    'b1_innovation_experiments.py': None,
    'b4p_anchor_verifier.py': 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    'qwen38_dual_task_experiments.py': 'Q38_RUNTIME_REVISION = "2026-08-24.official-evidence-scorer-v3"',
    'b4_q38_full_oof.py': 'Q38_OOF_REVISION = "2026-08-23.full64-three-fold-oof-v2"',
    'b4_task1_q38.py': 'TASK1_RUNTIME_REVISION = "2026-08-24.q38-full64-official-evidence-v4"',
    'b4e_evidence_set.py': 'B4E_RUNTIME_REVISION = "2026-08-24.official-one-to-one-event-set-v1"',
    'b4e_candidate_meta.py': 'save_candidate_audits: bool = True',
    'b4_final_submission.py': 'FINAL_RUNTIME_REVISION = "2026-08-25.no-retrain-threefold-test-v1"',
}
stale = []
for name, marker in MODULES.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('请一次性上传并覆盖：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded:
            raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

for path in [TRAIN_PATH, TEST_PATH, FOLD_REFERENCE, Q14_OOF, Q14_DECISION,
             Q38_FACTOR_OOF, Q38_FACTOR_CONFIG, TASK1_SELECTED_CONFIG]:
    assert path.exists(), path
sys.path.insert(0, str(ROOT))
print({'final_root': str(FINAL_ROOT), 'test': str(TEST_PATH)})


In [ ]:
#@title 2. 环境、Kernel 与模块硬检查
import numpy as np
import pandas as pd
import torch, transformers, sklearn, scipy, joblib
import b1_experiments as b1
import b1_innovation_experiments as inn
import b4p_anchor_verifier as b4
import qwen38_dual_task_experiments as q38
import b4_q38_full_oof as q38oof
import b4_task1_q38 as t1
import b4e_evidence_set as b4e
import b4e_candidate_meta as b4e2
import b4_final_submission as final
for module in (b1, inn, b4, q38, q38oof, t1, b4e, b4e2, final):
    importlib.reload(module)

assert sklearn.__version__ == '1.7.2', sklearn.__version__
assert final.FINAL_RUNTIME_REVISION == '2026-08-25.no-retrain-threefold-test-v1'
needs_gpu = bool(RUN_FACTOR_Q38_FOLDS or RUN_FACTOR_Q14_FOLDS or RUN_TASK1_FOLDS)
kernel_status = b4.qwen35_kernel_status()
print({
    'numpy': np.__version__, 'scipy': scipy.__version__,
    'pandas': pd.__version__, 'sklearn': sklearn.__version__,
    'transformers': transformers.__version__, 'torch': torch.__version__,
    'kernel': kernel_status,
})
if needs_gpu:
    assert torch.cuda.is_available()
    assert kernel_status['causal_conv1d']
    assert kernel_status['flash_linear_attention']
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    assert gpu_gb >= 70, f'需要 A100 80GB，当前 {gpu_gb:.1f} GB'
    torch.set_float32_matmul_precision('high')


In [ ]:
#@title 3. 数据、冻结配置与九个 adapter 完整性
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
test_corpus = b4.load_test_data(ROOT, TEST_PATH)
reference = np.load(FOLD_REFERENCE, allow_pickle=True)
assert reference['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = reference['folds'].astype(int)
assert len(test_corpus.texts) == 378
assert len(set(test_corpus.row_ids.astype(str))) == len(test_corpus.texts)

q14_payload = json.loads(Q14_DECISION.read_text(encoding='utf-8'))['config']
if q14_payload.get('lora_target_leaves') is not None:
    q14_payload['lora_target_leaves'] = tuple(q14_payload['lora_target_leaves'])
Q14_CFG = b4.B4PConfig(**q14_payload)
Q38_CFG = final.load_dataclass_config(
    Q38_FACTOR_CONFIG, b4.B4PConfig, tuple_fields=('lora_target_leaves',)
)
TASK1_CFG = final.load_dataclass_config(
    TASK1_SELECTED_CONFIG, t1.Task1Full64Config,
    tuple_fields=('candidate_caps_for_audit', 'lora_target_leaves'),
)
META_CFG = b4e2.CandidateMetaConfig()
assert Q14_CFG.verifier_model == 'Qwen/Qwen3-14B'
assert Q38_CFG.verifier_model == 'Qwen/Qwen3.8-27B'
assert TASK1_CFG.model_name == 'Qwen/Qwen3.8-27B'
assert TASK1_CFG.lora_last_n_layers is None
# Full64 是历史实验名；Fold-0 candidate-recall gate 最终冻结为 128。
# 这与三折 OOF 中每帖最多 128 个 evidence candidates 完全一致。
assert TASK1_CFG.validation_candidates_per_post == 128, (
    'Task1 candidate cap 与已确认 OOF 不一致：'
    f'{TASK1_CFG.validation_candidates_per_post} != 128'
)
print('Frozen Task1 candidate cap:', TASK1_CFG.validation_candidates_per_post)

def q14_adapter(fold):
    return Q14_ROOT / 'OOF' / f'fold_{fold}' / 'verifier' / 'adapter_final'
def q38_factor_adapter(fold):
    return Q38_FACTOR_ROOT / f'fold_{fold}' / 'verifier' / 'adapter_final'
def task1_fold_root(fold):
    if fold == 0:
        return TASK1_FOLD0_ROOT / 'Q38_FULL64' / 'fold_0'
    return TASK1_CONFIRM_ROOT / 'Q38_FULL64' / f'fold_{fold}'
def task1_proposer_checkpoint(fold):
    if fold == 0:
        root = TASK1_FOLD0_ROOT / 'MODERNBERT_PROPOSER'
    else:
        root = TASK1_CONFIRM_ROOT / 'MODERNBERT_PROPOSER'
    return root / 'checkpoints' / f'fold_{fold}.pt'

artifact_rows = []
for fold in range(3):
    paths = {
        'q14_factor_adapter': q14_adapter(fold) / 'adapter_config.json',
        'q38_factor_adapter': q38_factor_adapter(fold) / 'adapter_config.json',
        'task1_adapter': task1_fold_root(fold) / 'adapter' / 'adapter_final' / 'adapter_config.json',
        'task1_lexicon': task1_fold_root(fold) / 'evidence_lexicon.json',
        'task1_frozen_spec': task1_fold_root(fold) / 'frozen_task1_spec.json',
        'proposer_checkpoint': task1_proposer_checkpoint(fold),
        'task1_oof_audit': task1_fold_root(fold) / 'EVALUATION' / 'evidence_candidate_audit.csv',
        'task1_oof_predictions': task1_fold_root(fold) / 'EVALUATION' / 'validation_predictions.csv',
    }
    for kind, path in paths.items():
        artifact_rows.append({'fold': fold, 'kind': kind, 'exists': path.exists(), 'path': str(path)})
artifact_status = pd.DataFrame(artifact_rows)
display(artifact_status)
missing = artifact_status.loc[~artifact_status.exists]
if len(missing):
    raise FileNotFoundError('Final artifacts missing:\n' + missing.to_string(index=False))

# 三折 adapter 必须来自同一套已确认的 Task1 配置；只有 fold 字段允许不同。
frozen_fields = (
    'model_name', 'n_splits', 'max_length', 'context_chars',
    'candidate_max_chars', 'validation_candidates_per_post',
    'evidence_top_k', 'evidence_margin_threshold',
    'evidence_length_penalty', 'conditioned_risk_blend_weight',
    'lora_last_n_layers', 'attention_implementation',
    'qwen35_fa2_position_guard', 'require_qwen35_fast_kernels',
)
selected_payload = dataclasses.asdict(TASK1_CFG)
for fold in range(3):
    spec_path = task1_fold_root(fold) / 'frozen_task1_spec.json'
    spec_payload = json.loads(spec_path.read_text(encoding='utf-8'))
    spec_cfg = spec_payload.get('config', spec_payload)
    for field in frozen_fields:
        assert spec_cfg.get(field) == selected_payload[field], (
            f'Fold {fold} frozen Task1 config drift: {field}='
            f'{spec_cfg.get(field)!r}, expected {selected_payload[field]!r}'
        )
print({'train_rows': len(bundle.texts), 'test_rows': len(test_corpus.texts),
       'fold_sizes': np.bincount(folds).tolist(), 'artifact_gate': 'PASS'})


In [ ]:
#@title 4. 全三折 OOF 重拟合 B4-E2（CPU，数秒级）
fold_artifacts = {
    fold: (
        task1_fold_root(fold) / 'EVALUATION' / 'evidence_candidate_audit.csv',
        task1_fold_root(fold) / 'EVALUATION' / 'validation_predictions.csv',
    )
    for fold in range(3)
}
meta_model_path = META_FINAL_ROOT / 'B4E2_FINAL_ALL_OOF_META_CALIBRATOR.joblib'
if RUN_META_REFIT:
    meta_model, meta_candidate_table, meta_decision = final.fit_all_oof_meta_calibrator(
        fold_artifacts, bundle, folds, META_CFG, META_FINAL_ROOT
    )
    print(json.dumps(meta_decision, ensure_ascii=False, indent=2))
else:
    assert meta_model_path.exists(), meta_model_path
    meta_model = joblib.load(meta_model_path)
    print('[resume final meta]', meta_model_path)


## Factor test inference

下面只打分，不调用 Trainer。每折完成后会生成 `factor_test_logits.npz`；断线后重跑会直接复用。


In [ ]:
#@title 5. Factor 语义缓存（只在需要打分时建立）
train_corpus = b4.training_corpus(bundle)
q14_train_cache = q14_test_cache = q38_train_cache = q38_test_cache = None
if RUN_FACTOR_Q14_FOLDS:
    q14_train_cache = b4.prepare_semantic_cache(
        train_corpus, Q14_CFG, FINAL_ROOT / 'SEMANTIC_CACHE' / 'Q14_TRAIN'
    )
    q14_test_cache = b4.prepare_semantic_cache(
        test_corpus, Q14_CFG, FINAL_ROOT / 'SEMANTIC_CACHE' / 'Q14_TEST'
    )
if RUN_FACTOR_Q38_FOLDS:
    q38_train_cache = b4.prepare_semantic_cache(
        train_corpus, Q38_CFG, FINAL_ROOT / 'SEMANTIC_CACHE' / 'Q38_TRAIN'
    )
    q38_test_cache = b4.prepare_semantic_cache(
        test_corpus, Q38_CFG, FINAL_ROOT / 'SEMANTIC_CACHE' / 'Q38_TEST'
    )
print('Factor semantic caches ready.')


In [ ]:
#@title 6. Qwen3.8-27B Factor 三折 test 打分（可续跑）
for fold in RUN_FACTOR_Q38_FOLDS:
    print(f'\n========== FACTOR Q38 TEST FOLD {fold} ==========')
    started = time.perf_counter()
    final.score_factor_test_fold(
        q38_factor_adapter(fold), test_corpus, q38_test_cache,
        train_corpus, bundle, q38_train_cache, np.flatnonzero(folds != fold),
        Q38_CFG, FACTOR_TEST_ROOT / 'Q38' / f'fold_{fold}',
    )
    print({'fold': fold, 'elapsed_minutes_this_call': (time.perf_counter()-started)/60})
print('Q38 Factor scheduled folds complete.')


In [ ]:
#@title 7. Qwen3-14B Factor 三折 test 打分（可续跑）
for fold in RUN_FACTOR_Q14_FOLDS:
    print(f'\n========== FACTOR Q14 TEST FOLD {fold} ==========')
    started = time.perf_counter()
    final.score_factor_test_fold(
        q14_adapter(fold), test_corpus, q14_test_cache,
        train_corpus, bundle, q14_train_cache, np.flatnonzero(folds != fold),
        Q14_CFG, FACTOR_TEST_ROOT / 'Q14' / f'fold_{fold}',
    )
    print({'fold': fold, 'elapsed_minutes_this_call': (time.perf_counter()-started)/60})
print('Q14 Factor scheduled folds complete.')


In [ ]:
#@title 8. 冻结 25% Q14 + 75% Q38 Factor 输出
q14_test_paths = [FACTOR_TEST_ROOT / 'Q14' / f'fold_{fold}' / 'factor_test_logits.npz' for fold in range(3)]
q38_test_paths = [FACTOR_TEST_ROOT / 'Q38' / f'fold_{fold}' / 'factor_test_logits.npz' for fold in range(3)]
FACTOR_READY = all(path.exists() for path in q14_test_paths + q38_test_paths)
if FACTOR_READY:
    factor_frame, factor_decision = final.final_factor_predictions(
        bundle, folds, test_corpus, Q14_OOF, Q38_FACTOR_OOF,
        q14_test_paths, q38_test_paths, Q38_CFG, FACTOR_TEST_ROOT / 'FINAL',
        q38_weight=0.75,
    )
    display(factor_frame.head())
    print(json.dumps(factor_decision, indent=2))
else:
    print('Factor 尚缺：', [str(path) for path in q14_test_paths + q38_test_paths if not path.exists()])


## Task1 test inference

每折先对 test 运行已训练的 ModernBERT token proposer，再加载对应 Qwen3.8 adapter 完成 Document → Evidence → Conditioned Risk。所有 prompt margin 都按 chunk 续跑。


In [ ]:
#@title 9. ModernBERT token proposer 三折 test 推理
for fold in RUN_TASK1_FOLDS:
    output = TASK1_TEST_ROOT / f'fold_{fold}' / 'PROPOSER' / 'token_proposals_test.npz'
    print(f'\n========== PROPOSER TEST FOLD {fold} ==========')
    proposer = final.prepare_modernbert_proposer_test(
        test_corpus, task1_proposer_checkpoint(fold), fold, output, batch_size=32
    )
    print(proposer.metrics)


In [ ]:
#@title 10. Qwen3.8 Task1 三折 test 打分（最耗时，可续跑）
for fold in RUN_TASK1_FOLDS:
    print(f'\n========== TASK1 Q38 TEST FOLD {fold} ==========')
    fold_root = task1_fold_root(fold)
    adapter = fold_root / 'adapter' / 'adapter_final'
    lexicon = t1.EvidenceLexicon.from_json(
        json.loads((fold_root / 'evidence_lexicon.json').read_text(encoding='utf-8'))
    )
    proposer = final.prepare_modernbert_proposer_test(
        test_corpus, task1_proposer_checkpoint(fold), fold,
        TASK1_TEST_ROOT / f'fold_{fold}' / 'PROPOSER' / 'token_proposals_test.npz',
        batch_size=32,
    )
    cfg = dataclasses.replace(TASK1_CFG, fold=fold)
    started = time.perf_counter()
    outputs = final.run_task1_test_fold(
        test_corpus, adapter, lexicon, cfg, proposer,
        TASK1_TEST_ROOT / f'fold_{fold}',
    )
    print({**outputs, 'elapsed_minutes_this_call': (time.perf_counter()-started)/60})
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
#@title 11. 三折 Task1 融合 + 最终 B4-E2 Evidence
task1_test_dirs = [TASK1_TEST_ROOT / f'fold_{fold}' for fold in range(3)]
TASK1_READY = all(
    (path / 'q38_task1_test_outputs.npz').exists() and
    (path / 'evidence_candidate_audit.csv').exists()
    for path in task1_test_dirs
)
if TASK1_READY:
    if 'meta_model' not in globals():
        assert meta_model_path.exists(), meta_model_path
        meta_model = joblib.load(meta_model_path)
    task1_frame, task1_decision = final.assemble_task1_test_predictions(
        test_corpus, task1_test_dirs, meta_model, META_CFG, TASK1_CFG,
        TASK1_TEST_ROOT / 'FINAL',
    )
    display(task1_frame.head())
    print(json.dumps(task1_decision, ensure_ascii=False, indent=2))
else:
    print('Task1 尚缺：', [str(path) for path in task1_test_dirs
                              if not (path / 'q38_task1_test_outputs.npz').exists()])


In [ ]:
#@title 12. 生成并硬审官方 Lenormand.csv
factor_path = FACTOR_TEST_ROOT / 'FINAL' / 'factor_predictions.csv'
task1_path = TASK1_TEST_ROOT / 'FINAL' / 'task1_test_predictions.csv'
if RUN_FINAL_ASSEMBLY and factor_path.exists() and task1_path.exists():
    factor_frame = pd.read_csv(factor_path, keep_default_na=False)
    task1_frame = pd.read_csv(task1_path, keep_default_na=False)
    submission_path = SUBMISSION_ROOT / 'Lenormand.csv'
    submission_audit = final.build_and_audit_submission(
        test_corpus, task1_frame, factor_frame, submission_path
    )
    display(pd.read_csv(submission_path, keep_default_na=False).head(10))
    print(json.dumps(submission_audit, ensure_ascii=False, indent=2))
else:
    print({'factor_ready': factor_path.exists(), 'task1_ready': task1_path.exists()})


In [ ]:
#@title 13. 输出分布审计与轻量备份包
submission_path = SUBMISSION_ROOT / 'Lenormand.csv'
if submission_path.exists():
    submission = pd.read_csv(submission_path, keep_default_na=False)
    submission['factor_count'] = submission.factors.map(lambda value: len(__import__('ast').literal_eval(value)))
    submission['evidence_count'] = submission.evidence.map(
        lambda value: len([x for x in str(value).split(';') if x.strip()])
    )
    display(submission.groupby('risk_level').agg(
        rows=('row_id', 'size'), mean_evidence=('evidence_count', 'mean'),
        mean_factors=('factor_count', 'mean')
    ).reindex(final.OFFICIAL_RISKS))
    if not 2.0 <= submission.factor_count.mean() <= 5.0:
        print('WARNING: test Factor cardinality 偏离 OOF 常见范围，上传前人工复核。')

    report_root = SUBMISSION_ROOT / 'UPLOAD_PACKAGE'
    report_root.mkdir(parents=True, exist_ok=True)
    for source in [
        submission_path, SUBMISSION_ROOT / 'FINAL_SUBMISSION_AUDIT.json',
        FACTOR_TEST_ROOT / 'FINAL' / 'factor_test_decision.json',
        TASK1_TEST_ROOT / 'FINAL' / 'task1_test_decision.json',
        META_FINAL_ROOT / 'B4E2_FINAL_REFIT_DECISION.json',
        ROOT / 'b4_final_submission.py',
    ]:
        if source.exists():
            shutil.copy2(source, report_root / source.name)
    archive = shutil.make_archive('/content/Lenormand_B4_FINAL_UPLOAD', 'zip', report_root)
    print('CSV ready:', submission_path)
    print('Light package:', archive)
    # files.download(str(submission_path))
else:
    print('Lenormand.csv 尚未生成。')


## 最终上传前纪律

- 只上传 `SUBMISSION/Lenormand.csv`；Evidence 已按官网要求使用分号分隔。
- 不再使用 leaderboard 反馈调 `.40 / gap40 / k0232 / 0.75 blend / Factor thresholds`。
- `FINAL_SUBMISSION_AUDIT.json` 必须显示 `READY_TO_UPLOAD`、`verbatim_failures=0`、`indicator_nonempty_evidence=0`。
- 当前 notebook 处理的是 378 条 live leaderboard data；主办方最终 held-out 数据到达后，只需替换 `TEST_PATH` 并重跑 test 缓存目录。
